# Aula 8 · Treinar, medir e levar pro campo

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tassid/pdi-agro/blob/main/notebooks/Aula8_YOLO_Treino_Avaliacao.ipynb)

Material do módulo YOLO do **Pixel no Campo** (PPGTCA/UTFPR Medianeira). Companheiro dos labs 8.1 a 8.4 do site.

**Roteiro do notebook:**

1. Gerar um banco de piquetes vistos de drone: 10 voos na fazenda A (nelore em pasto verde) e 4 voos na fazenda B (gado escuro, solo exposto, cupinzeiros), **separados por voo**.
2. Treinar um `yolo26n` só com a fazenda A (rode antes do intervalo).
3. Avaliar no teste da fazenda A e na fazenda B: mAP50, mAP50-95, P, R e erro de contagem.
4. Escolher o limiar de confiança e colocar um pouco da fazenda B no treino.
5. Contar bois passando num brete com rastreamento.
6. Exportar o modelo.

In [ ]:
# Ambiente de execução → Alterar o tipo de ambiente → GPU (T4). Depois rode esta célula.
!pip install -q -U ultralytics lap onnx onnxruntime   # lap: rastreador (parte 5) · onnx: exportação (parte 6)
import ultralytics
ultralytics.checks()

In [ ]:
import cv2, numpy as np, matplotlib.pyplot as plt

def mostrar(*imgs, titulos=None, tam=5):
    """mostra imagens BGR (do OpenCV) lado a lado"""
    fig, ax = plt.subplots(1, len(imgs), figsize=(tam * len(imgs), tam))
    ax = np.atleast_1d(ax)
    for i, im in enumerate(imgs):
        ax[i].imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); ax[i].axis("off")
        if titulos: ax[i].set_title(titulos[i])
    plt.show()

## 1. O banco

O gerador é o mesmo da Aula 7. Cada **voo** tem a sua luz e o seu sol; todas as fotos de um voo vão para o mesmo conjunto. É o split por grupo da Aula 5.

| conjunto | de onde vem | uso |
|---|---|---|
| `train` | fazenda A, voos 1 a 6 | treino |
| `val` | fazenda A, voos 7 e 8 | escolher a época (best.pt) e o limiar |
| `test_A` | fazenda A, voos 9 e 10 | teste no mesmo domínio |
| `extra_B` | fazenda B, voos 1 e 2 | só na parte 4 |
| `test_B` | fazenda B, voos 3 e 4 | teste em outro domínio |

In [ ]:
import numpy as np, cv2

def _ruido(rng, h, w, escala):
    """ruído suave: sorteia uma grade pequena e amplia com interpolação cúbica"""
    g = rng.random((max(2, h // escala), max(2, w // escala))).astype(np.float32)
    return cv2.resize(g, (w, h), interpolation=cv2.INTER_CUBIC).clip(0, 1)

def gerar_piquete(semente, fazenda="A", n=None, tam=640, luz=1.0, sol=(5, 4)):
    """Gera um piquete visto de drone e as caixas verdadeiras dos animais.
    fazenda A: nelore claro em pasto verde.  fazenda B: gado escuro, solo exposto, mais cupinzeiros.
    Devolve (imagem BGR uint8, lista de caixas [x0, y0, x1, y1] em pixels)."""
    rng = np.random.default_rng(semente)
    H = W = tam
    B = fazenda == "B"
    n = n if n is not None else int(rng.integers(8, 22))
    # capim: verde com manchas secas
    n1, n2 = _ruido(rng, H, W, 80), _ruido(rng, H, W, 12)
    seco = np.clip((_ruido(rng, H, W, 140) - (0.35 if B else 0.55)) * 3, 0, 1)
    img = np.stack([52 + 18 * n1 + 8 * n2 + 10 * seco,            # B
                    118 + 36 * n1 + 20 * n2 + 30 * seco,           # G
                    78 + 40 * n1 + 18 * n2 + 70 * seco], -1)       # R
    if B:  # solo vermelho aparecendo
        solo = np.clip((_ruido(rng, H, W, 60) - 0.6) * 5, 0, 1)[..., None]
        img = img * (1 - solo) + np.array([48, 74, 150]) * solo
    img = img.astype(np.float32)
    sx, sy = sol
    # árvores (sombra agora, copa por último)
    arvores = [(int(rng.integers(60, W - 60)), int(rng.integers(60, H - 60)), int(rng.integers(40, 60))) for _ in range(int(rng.integers(1, 3)))]
    sombra = np.zeros((H, W), np.uint8)
    for x, y, r in arvores:
        cv2.circle(sombra, (x + 4 * sx, y + 4 * sy), r, 1, -1)
    # cupinzeiros
    cupins = []
    for _ in range(int(rng.integers(9, 13)) if B else int(rng.integers(3, 5))):
        cupins.append((int(rng.integers(20, W - 20)), int(rng.integers(20, H - 20)), int(rng.integers(7, 11))))
    for x, y, r in cupins:
        cv2.circle(sombra, (x + sx // 2, y + sy // 2), r, 1, -1)
    # animais
    animais, tent = [], 0
    while len(animais) < n and tent < 2000:
        tent += 1
        x, y = rng.uniform(28, W - 28), rng.uniform(28, H - 28)
        bez = rng.random() < 0.18
        a = (10.5 if bez else 16) + rng.random() * 2.4
        b = (5.2 if bez else 7.2) + rng.random() * 1.2
        if any(np.hypot(o[0] - x, o[1] - y) < 34 for o in animais):
            continue
        if any(np.hypot(c[0] - x, c[1] - y) < 26 for c in cupins):
            continue
        ang = rng.random() * 360
        escuro = B and rng.random() < 0.6     # na fazenda A só tem nelore claro
        cor = (np.array([30, 40, 58]) if rng.random() < 0.5 else np.array([36, 38, 44])) if escuro else np.array([206, 218, 224]) + rng.random(3) * 16
        animais.append((x, y, a, b, ang, cor))
    for x, y, a, b, ang, cor in animais:
        cv2.ellipse(sombra, (int(x + sx), int(y + sy)), (int(a), int(b)), ang, 0, 360, 1, -1)
    img[sombra > 0] *= 0.56
    for x, y, r in cupins:
        cv2.circle(img, (x, y), r, (72, 98, 138) if B else (132, 160, 176), -1)
    caixas = []
    for x, y, a, b, ang, cor in animais:
        cv2.ellipse(img, (int(x), int(y)), (int(a), int(b)), ang, 0, 360, cor.tolist(), -1)
        t = np.deg2rad(ang)
        hx, hy = x + 1.1 * a * np.cos(t), y + 1.1 * a * np.sin(t)
        cv2.ellipse(img, (int(hx), int(hy)), (int(0.3 * a) + 1, int(0.55 * b) + 1), ang, 0, 360, (cor * 0.92).tolist(), -1)
        # caixa: elipse do corpo com a cabeça até 1,4·a à frente
        ex = np.sqrt((1.4 * a * np.cos(t)) ** 2 + (b * np.sin(t)) ** 2)
        ey = np.sqrt((1.4 * a * np.sin(t)) ** 2 + (b * np.cos(t)) ** 2)
        caixas.append([max(0, x - ex), max(0, y - ey), min(W, x + ex), min(H, y + ey)])
    # copa das árvores por cima (animal debaixo fica parcialmente escondido, mas continua rotulado)
    copa = np.zeros((H, W), np.uint8)
    for x, y, r in arvores:
        cv2.circle(copa, (x, y), int(r * 0.7), 1, -1)
    folhas = _ruido(rng, H, W, 6)
    m = copa > 0
    img[m] = np.stack([30 + 14 * folhas, 72 + 34 * folhas, 34 + 30 * folhas], -1)[m]
    # quanto de cada animal ficou visível? some do rótulo se a copa cobrir mais de 70%
    vis = []
    for (x0, y0, x1, y1) in caixas:
        sub = copa[int(y0):int(y1) + 1, int(x0):int(x1) + 1]
        vis.append(1 - sub.mean() if sub.size else 1)
    caixas = [c for c, v in zip(caixas, vis) if v > 0.3]
    img = img * luz + rng.normal(0, 3, img.shape)
    return np.clip(img, 0, 255).astype(np.uint8), caixas

def linhas_yolo(caixas, W, H, classe=0):
    out = []
    for x0, y0, x1, y1 in caixas:
        out.append(f"{classe} {(x0 + x1) / 2 / W:.6f} {(y0 + y1) / 2 / H:.6f} {(x1 - x0) / W:.6f} {(y1 - y0) / H:.6f}")
    return out

def desenhar_rotulos(img, txt, cor=(0, 224, 255)):
    """lê um .txt no formato YOLO e desenha as caixas de volta na imagem (o teste que todo banco deve passar)"""
    H, W = img.shape[:2]
    out = img.copy()
    for linha in open(txt).read().split("\n"):
        if not linha.strip():
            continue
        c, cx, cy, w, h = map(float, linha.split())
        x0, y0, x1, y1 = (cx - w / 2) * W, (cy - h / 2) * H, (cx + w / 2) * W, (cy + h / 2) * H
        cv2.rectangle(out, (int(x0), int(y0)), (int(x1), int(y1)), cor, 2)
    return out

In [ ]:
import os, shutil, glob
RAIZ = "pasto"
shutil.rmtree(RAIZ, ignore_errors=True)
FOTOS_POR_VOO = 30

def gerar_voo(fazenda, voo, conjunto):
    rng = np.random.default_rng(1000 * (fazenda == "B") + voo)
    luz = rng.uniform(0.8, 1.15)                          # nublado, sol forte...
    sol = (int(rng.integers(2, 8)), int(rng.integers(2, 7)))   # comprimento da sombra muda com a hora
    os.makedirs(f"{RAIZ}/images/{conjunto}", exist_ok=True)
    os.makedirs(f"{RAIZ}/labels/{conjunto}", exist_ok=True)
    for k in range(FOTOS_POR_VOO):
        img, caixas = gerar_piquete(100_000 * (fazenda == "B") + 1000 * voo + k, fazenda, luz=luz, sol=sol)
        nome = f"{fazenda}_voo{voo:02d}_{k:04d}"
        cv2.imwrite(f"{RAIZ}/images/{conjunto}/{nome}.jpg", img, [cv2.IMWRITE_JPEG_QUALITY, 92])
        open(f"{RAIZ}/labels/{conjunto}/{nome}.txt", "w").write("\n".join(linhas_yolo(caixas, img.shape[1], img.shape[0])) + "\n")

for v in range(1, 11):
    gerar_voo("A", v, "train" if v <= 6 else "val" if v <= 8 else "test_A")
for v in range(1, 5):
    gerar_voo("B", v, "extra_B" if v <= 2 else "test_B")

def yaml(nome, train, val, test):
    open(f"{RAIZ}/{nome}", "w").write(f"""path: {os.path.abspath(RAIZ)}
train: {train}
val: {val}
test: {test}
names:
  0: bovino
""")

yaml("data.yaml", "images/train", "images/val", "images/test_A")
yaml("teste_A.yaml", "images/train", "images/val", "images/test_A")
yaml("teste_B.yaml", "images/train", "images/val", "images/test_B")
yaml("data_AB.yaml", "[images/train, images/extra_B]", "images/val", "images/test_B")

for c in ["train", "val", "test_A", "extra_B", "test_B"]:
    n_img = len(glob.glob(f"{RAIZ}/images/{c}/*.jpg"))
    n_box = sum(len(open(t).read().split()) // 5 for t in glob.glob(f"{RAIZ}/labels/{c}/*.txt"))
    print(f"{c:8s} {n_img:4d} imagens  {n_box:5d} caixas")
print(open(f"{RAIZ}/data.yaml").read())

### O teste do doutor do rótulo (Lab 8.1)

Antes de treinar, desenhe alguns rótulos de volta na imagem. Um formato trocado não dá erro: dá mAP perto de zero.

In [ ]:
amostras = sorted(glob.glob(f"{RAIZ}/images/train/*.jpg"))[:2] + sorted(glob.glob(f"{RAIZ}/images/test_B/*.jpg"))[:2]
mostrar(*[desenhar_rotulos(cv2.imread(f), f.replace("images", "labels").replace(".jpg", ".txt")) for f in amostras],
        titulos=["treino A", "treino A", "teste B", "teste B"], tam=4)

## 2. Treinar

Partimos dos pesos do COCO (transfer learning). Numa T4, 40 épocas levam uns 5 a 10 minutos. **Rode e vá para o intervalo.**

Sobre o aumento de dados: visto de cima, um boi de ponta-cabeça continua sendo boi, então ligamos o espelhamento vertical além do horizontal. Rotações arbitrárias (`degrees`) aumentam as caixas (a caixa de uma caixa girada é maior que o objeto), então ficam desligadas.

In [ ]:
from ultralytics import YOLO
EPOCAS = 40            # sem GPU? use 3 só para ver funcionando

modelo = YOLO("yolo26n.pt")
modelo.train(data=f"{RAIZ}/data.yaml", epochs=EPOCAS, imgsz=640, batch=16, patience=15,
             fliplr=0.5, flipud=0.5, mosaic=1.0, name="pasto_n", exist_ok=True, plots=True)
pasta = str(modelo.trainer.save_dir)      # onde o ultralytics salvou tudo (pesos, curvas, results.csv)
print("resultados em", pasta)

In [ ]:
import pandas as pd
r = pd.read_csv(f"{pasta}/results.csv"); r.columns = r.columns.str.strip()
print(r[["epoch", "train/box_loss", "val/box_loss", "metrics/precision(B)", "metrics/recall(B)", "metrics/mAP50(B)", "metrics/mAP50-95(B)"]].tail(8).to_string(index=False))
mostrar(cv2.imread(f"{pasta}/results.png"), tam=14)

**Leia as curvas:** se `train/box_loss` cai e `val/box_loss` sobe, o modelo está decorando os voos de treino. O `best.pt` é a época com o melhor mAP na validação.

## 3. Medir: fazenda A e fazenda B

`model.val()` calcula precisão, revocação, mAP50 e mAP50-95 com o mesmo casamento por IoU do Lab 8.3. Para o produtor, também importa o **erro de contagem**, que calculamos à parte.

In [ ]:
melhor = YOLO(f"{pasta}/weights/best.pt")

def erro_contagem(modelo, conjunto, conf):
    erros = []
    for img in sorted(glob.glob(f"{RAIZ}/images/{conjunto}/*.jpg")):
        n_pred = len(modelo.predict(img, conf=conf, verbose=False)[0].boxes)
        n_real = len(open(img.replace("images", "labels").replace(".jpg", ".txt")).read().split()) // 5
        erros.append(n_pred - n_real)
    erros = np.array(erros)
    return np.abs(erros).mean(), erros.mean()

def avaliar(modelo, conf=0.25):
    linhas = []
    for nome, y, conj in [("fazenda A (mesmo domínio)", "teste_A.yaml", "test_A"), ("fazenda B (outro domínio)", "teste_B.yaml", "test_B")]:
        m = modelo.val(data=f"{RAIZ}/{y}", split="test", conf=0.001, iou=0.7, plots=False, verbose=False)
        ea, vies = erro_contagem(modelo, conj, conf)
        linhas.append({"teste": nome, "mAP50": m.box.map50, "mAP50-95": m.box.map, "P": m.box.mp, "R": m.box.mr,
                       f"erro absoluto de contagem (conf={conf})": ea, "viés (+ conta a mais)": vies})
    return pd.DataFrame(linhas).round(3)

aval_A = avaliar(melhor)
aval_A

**O que vimos nos nossos testes:** o mAP50 quase não mudou entre as fazendas (0,99 contra 0,98), o mAP50-95 caiu um pouco (0,63 contra 0,58), mas o erro de contagem com `conf=0.25` quase triplicou: de uns 0,4 animal por foto na fazenda A para 1 na fazenda B, contando a mais (cupinzeiro virando boi). Os seus números vão variar um pouco. É a pegadinha do Lab 8.3: **o mAP esconde o que a contagem mostra.**

**Pra discutir (Lab 8.3):** o que caiu mais na fazenda B, a precisão ou a revocação? O viés de contagem é positivo (cupinzeiro contado como boi) ou negativo (boi escuro perdido)? Os dois erros podem se cancelar e esconder um modelo ruim.

### Escolher o limiar de confiança

O mAP não depende do `conf`; a contagem depende. Escolha o `conf` que maximiza o F1 **na validação** (nunca no teste).

In [ ]:
import torch
from ultralytics.utils.metrics import box_iou

def f1_por_conf(modelo, conjunto="val", confs=np.arange(0.05, 0.96, 0.05)):
    dados = []
    for img in sorted(glob.glob(f"{RAIZ}/images/{conjunto}/*.jpg")):
        r = modelo.predict(img, conf=0.01, verbose=False)[0]
        H, W = r.orig_shape
        txt = img.replace("images", "labels").replace(".jpg", ".txt")
        v = np.array([list(map(float, l.split()))[1:] for l in open(txt).read().splitlines() if l.strip()]).reshape(-1, 4)
        gt = np.stack([(v[:, 0] - v[:, 2] / 2) * W, (v[:, 1] - v[:, 3] / 2) * H, (v[:, 0] + v[:, 2] / 2) * W, (v[:, 1] + v[:, 3] / 2) * H], 1) if len(v) else np.zeros((0, 4))
        dados.append((r.boxes.xyxy.cpu(), r.boxes.conf.cpu().numpy(), torch.tensor(gt, dtype=torch.float32)))
    saida = []
    for c in confs:
        tp = fp = fn = 0
        for xyxy, cf, gt in dados:
            p = xyxy[cf >= c]
            if len(gt) == 0: fp += len(p); continue
            if len(p) == 0: fn += len(gt); continue
            m = box_iou(p, gt).numpy()
            usado = np.zeros(len(gt), bool); t = 0
            for i in np.argsort(-cf[cf >= c]):          # casamento guloso, maior confiança primeiro
                cand = np.where(usado, -1, m[i]); j = int(np.argmax(cand))
                if cand[j] >= 0.5: usado[j] = True; t += 1
            tp += t; fp += len(p) - t; fn += len(gt) - t
        P = tp / max(1, tp + fp); R = tp / max(1, tp + fn)
        saida.append(dict(conf=round(c, 2), P=P, R=R, F1=2 * P * R / max(1e-9, P + R)))
    return pd.DataFrame(saida).round(3)

tab = f1_por_conf(melhor)
conf_ideal = float(tab.loc[tab.F1.idxmax(), "conf"])
print(tab.to_string(index=False)); print("\nconf com maior F1 na validação:", conf_ideal)
plt.plot(tab.conf, tab.F1, label="F1"); plt.plot(tab.conf, tab.P, label="P"); plt.plot(tab.conf, tab.R, label="R")
plt.xlabel("conf"); plt.legend(); plt.grid(alpha=0.3); plt.show()

In [ ]:
avaliar(melhor, conf=conf_ideal)

## 4. Um pouco da fazenda B no treino

Quase sempre, um pouco de dado do domínio novo vale mais que muitas épocas a mais. Treinamos de novo com os voos 1 e 2 da fazenda B junto (`data_AB.yaml`) e testamos nos voos 3 e 4, que o modelo nunca viu.

In [ ]:
modelo_AB = YOLO("yolo26n.pt")
modelo_AB.train(data=f"{RAIZ}/data_AB.yaml", epochs=EPOCAS, imgsz=640, batch=16, patience=15,
                fliplr=0.5, flipud=0.5, name="pasto_AB", exist_ok=True, plots=False)
melhor_AB = YOLO(f"{modelo_AB.trainer.save_dir}/weights/best.pt")
comp = pd.concat([avaliar(melhor, conf_ideal).assign(treino="só A"), avaliar(melhor_AB, conf_ideal).assign(treino="A + 2 voos de B")])
comp

## 5. Contar bois no brete (Lab 8.4)

Geramos um vídeo curto de um corredor de curral visto de cima, com uma trave da porteira escondendo os animais por alguns quadros. O rastreador ByteTrack do ultralytics dá um número (id) a cada animal; comparamos duas formas de contar: quantos ids foram criados e quantas trilhas cruzaram uma linha **depois** da trave.

In [ ]:
def video_brete(caminho, n=10, semente=3, W=640, H=360, trave=True):
    rng = np.random.default_rng(semente)
    bois, t = [], 0
    for _ in range(n):
        t += rng.uniform(25, 60)
        escuro = rng.random() < 0.3
        bois.append(dict(t0=t, v=rng.uniform(3.2, 4.6), y=H / 2 + rng.uniform(-25, 25), a=rng.uniform(32, 38), b=rng.uniform(14, 17),
                         cor=(40, 52, 80) if escuro else (206, 218, 226)))
    fim = int(bois[-1]["t0"] + (W + 120) / 3.2)
    out = cv2.VideoWriter(caminho, cv2.VideoWriter_fourcc(*"mp4v"), 25, (W, H))
    passaram = []
    for f in range(fim):
        img = np.full((H, W, 3), (98, 122, 138), np.uint8)
        img[:70] = img[-70:] = (73, 93, 107)
        for bo in bois:
            x = -60 + (f - bo["t0"]) * bo["v"]
            if -60 < x < W + 60:
                cv2.ellipse(img, (int(x + 6), int(bo["y"] + 6)), (int(bo["a"]), int(bo["b"])), 0, 0, 360, (60, 70, 80), -1)
                cv2.ellipse(img, (int(x), int(bo["y"])), (int(bo["a"]), int(bo["b"])), 0, 0, 360, bo["cor"], -1)
                cv2.ellipse(img, (int(x + 1.1 * bo["a"]), int(bo["y"])), (int(0.3 * bo["a"]), int(0.55 * bo["b"])), 0, 0, 360, bo["cor"], -1)
        if trave:
            cv2.rectangle(img, (200 - 22, 50), (200 + 22, H - 50), (38, 58, 74), -1)   # trave em x = 200
        out.write(np.clip(img + rng.normal(0, 3, img.shape), 0, 255).astype(np.uint8))
    out.release()
    return n

n_real = video_brete("brete.mp4")
print("bois no vídeo:", n_real)

In [ ]:
LINHA = 440      # linha de contagem, depois da trave

def contar(modelo, video):
    ultimo_x, contados, ids = {}, set(), set()
    for r in modelo.track(video, tracker="bytetrack.yaml", conf=conf_ideal, stream=True, verbose=False, persist=True):
        if r.boxes.id is None: continue
        for i, b in zip(r.boxes.id.int().tolist(), r.boxes.xyxy.tolist()):
            ids.add(i); x = (b[0] + b[2]) / 2
            if i in ultimo_x and ultimo_x[i] < LINHA <= x: contados.add(i)
            ultimo_x[i] = x
    return len(contados), len(ids)

c, i = contar(YOLO(f"{pasta}/weights/best.pt"), "brete.mp4")
print(f"bois de verdade: {n_real} · contados na linha: {c} · ids criados: {i}")

Nos nossos testes deu **10 na linha e 25 ids**: os animais perderam o número ao passar atrás da trave e ganharam outro do lado de lá. Contar ids inflaria o rebanho em 2,5 vezes; contar o cruzamento da linha acertou.

**Pra testar:** mude `LINHA` para 200 (bem em cima da trave) e rode de novo. A contagem cai para perto de zero, porque o animal some antes da linha e reaparece já do outro lado, com outro id. No campo, a linha vai onde a câmera enxerga bem; e é para casos assim que serve a memória do rastreador (`track_buffer` no arquivo `bytetrack.yaml`, o slider do Lab 8.4).

Se o modelo treinado em piquetes não achar os bois no brete (o ângulo e o tamanho são outros), isso é mais um exemplo de mudança de domínio: o que fazer? Rotular alguns quadros do brete e treinar de novo.

Para quem quiser a solução pronta do ultralytics: `from ultralytics import solutions; solutions.ObjectCounter(model=..., region=[(320, 0), (320, 360)])`.

## 6. Exportar

`.pt` só roda com PyTorch. Para servidor, celular ou placa embarcada, exporte.

In [ ]:
caminho_onnx = melhor.export(format="onnx", imgsz=640)
print("salvo em", caminho_onnx)
r = YOLO(caminho_onnx, task="detect").predict(sorted(glob.glob(f"{RAIZ}/images/test_A/*.jpg"))[0], verbose=False)[0]
print("o modelo ONNX achou", len(r.boxes), "animais")
# celular / Raspberry Pi: melhor.export(format="tflite")

## 7. Opcional: o banco do caso real (GWHD)

O `GlobalWheat2020.yaml` vem com o ultralytics e baixa sozinho na primeira vez (alguns gigabytes). Não rode em sala. O treino é com fotos da Europa e do Canadá; o teste, com fotos da Austrália, do Japão e da China.

In [ ]:
# trigo = YOLO("yolo26n.pt")
# trigo.train(data="GlobalWheat2020.yaml", epochs=50, imgsz=1024, batch=8)
# trigo.val(split="test")

## Desafios

1. **Aquecimento.** Treine de novo com `imgsz=320`. Compare mAP50, mAP50-95 e erro de contagem. Explique com o Lab 8.2 (os bezerros!).
2. **Médio.** Troque o split por voo por um sorteio foto a foto (misture todas as fotos da fazenda A e separe 20% ao acaso). O mAP no teste sobe? Isso quer dizer que o modelo ficou melhor?
3. **Desafio.** Rotule 50 fotos reais suas no CVAT ou no Roboflow (aviário, pomar, bandeja de sementes), treine um `yolo26n` com split por dia de coleta e reporte mAP50, mAP50-95 e erro de contagem.